In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

data = pd.read_csv("all_participants_data.csv", index_col="Participant")

indexes = data.index

#getting list of median arousals and valences

allMedianArousals = list(data["median_arousal"])
allMedianValences = list(data["median_valence"])

#getting median of the rows

medianArousal = data["median_arousal"].median()
medianValence = data["median_valence"].median()

data = data.drop(columns=['median_arousal', 'median_valence'])

scaler = StandardScaler()
scaledArray = scaler.fit_transform(data)
scaledData = pd.DataFrame(scaledArray, index=indexes)



#getting percentile classes for arousals and valences

arousalPercentiles = np.percentile(allMedianArousals, [20, 40, 60, 80])
arousalPercentileClasses = []

arousalPercentileClasses.append([(row, 0, index) for j, (index, row) in enumerate(scaledData.iterrows()) if allMedianArousals[j] <= arousalPercentiles[0]])
for i in range(0, 3):
    arousalPercentileClasses.append([(row, i+1, index) for j, (index, row) in enumerate(scaledData.iterrows()) if allMedianArousals[j] > arousalPercentiles[i] and allMedianArousals[j] <= arousalPercentiles[i+1]])
arousalPercentileClasses.append([(row, 4, index) for j, (index, row) in enumerate(scaledData.iterrows()) if allMedianArousals[j] > arousalPercentiles[3]])

arousalPercentileClasses = sum(arousalPercentileClasses, [])


valencePercentiles = np.percentile(allMedianValences, [20, 40, 60, 80])
valencePercentileClasses = []

valencePercentileClasses.append([(row, 0, index) for j, (index, row) in enumerate(scaledData.iterrows()) if allMedianValences[j] <= valencePercentiles[0]])
for i in range(0, 3):
    valencePercentileClasses.append([(row, i+1, index) for j, (index, row) in enumerate(scaledData.iterrows()) if allMedianValences[j] > valencePercentiles[i] and allMedianValences[j] <= valencePercentiles[i+1]])
valencePercentileClasses.append([(row, 4, index) for j, (index, row) in enumerate(scaledData.iterrows()) if allMedianValences[j] > valencePercentiles[3]])

valencePercentileClasses = sum(valencePercentileClasses, [])

In [ ]:
import mord 
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

model1 = mord.LogisticIT()

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

class OrdinalNN(nn.Module):
    def __init__(self, input_dim, hidden_dim1, hidden_dim2, num_classes):
        super(OrdinalNN, self).__init__()
        
        layers = []
        layers.append(nn.Linear(input_dim, hidden_dim1))
        layers.append(nn.ReLU())
        
        layers.append(nn.Linear(hidden_dim1, hidden_dim2))
        layers.append(nn.ReLU())
        
        self.hidden = nn.Sequential(*layers)
        self.output = nn.Linear(hidden_dim2, num_classes - 1) 
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, x):
        x = self.hidden(x)
        logits = self.output(x)
        cum_probs = self.sigmoid(logits) 
        return cum_probs
    
model2 = OrdinalNN(288, 16, 8, 5)

In [ ]:
class OrdinalLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim, K, num_classes):
        super(OrdinalLSTM, self).__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers=K, batch_first=True)
        self.fc = nn.Linear(hidden_dim, num_classes - 1)  # K-1 cumulative probabilities
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, x):
        out, _ = self.lstm(x)      
        out = out[:, -1, :]     
        logits = self.fc(out)
        cum_probs = self.sigmoid(logits) 
        return cum_probs

model3 = OrdinalLSTM(288, 16, 2, 5)





In [ ]:
from scipy.stats import pearsonr
import numpy as np

def CCcoefficient(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    mean_true = np.mean(y_true)
    mean_pred = np.mean(y_pred)
    var_true = np.var(y_true)
    var_pred = np.var(y_pred)
    cov = np.mean((y_true - mean_true) * (y_pred - mean_pred))

    ccc = (2 * cov) / (var_true + var_pred + (mean_true - mean_pred) ** 2)
    return ccc

In [ ]:
indexes = data.index

participants = list(set(indexes))

X = [element[0] for element in arousalPercentileClasses]
Y = [element[1] for element in arousalPercentileClasses]
orderedIndexes = [element[2] for element in arousalPercentileClasses]

results = []


for participant in participants:
    trainIdx = [p for p in participants if p != participant]
    testIdx = participant

    Xtrain, Ytrain = [x for i, x in enumerate(X) if orderedIndexes[i] != participant], [x for i, x in enumerate(Y) if orderedIndexes[i] != participant]
    Xtest, Ytest = [x for i, x in enumerate(X) if orderedIndexes[i] == participant], [x for i, x in enumerate(Y) if orderedIndexes[i] == participant]

    model1.fit(np.array(Xtrain), np.array(Ytrain))

    Ypred = model1.predict_proba(np.array(Xtest))
    Ypred = [np.argmax(x) for x in Ypred]

    results.append((pearsonr(Ytest, Ypred)[0], CCcoefficient(Ytest, Ypred)))

print(results)

In [ ]:
def cum_probs_to_labels(cum_probs):
    """
    cum_probs: Tensor of shape (batch_size, K-1)
    Returns: Tensor of predicted class labels (0-indexed)
    """
    # Threshold at 0.5
    preds = (cum_probs >= 0.5).float()
    # Sum over thresholds to get class index
    labels = preds.sum(dim=1).long()
    return labels

In [ ]:
y_true_all = []
y_pred_all = []

indexes = data.index

participants = list(set(indexes))

X = [element[0] for element in valencePercentileClasses]
Y = [element[1] for element in valencePercentileClasses]
orderedIndexes = [element[2] for element in valencePercentileClasses]

results = []

for participant in participants:
    
    trainIdx = [p for p in participants if p != participant]
    testIdx = participant

    Xtrain, Ytrain = [x for i, x in enumerate(X) if orderedIndexes[i] != participant], [x for i, x in enumerate(Y) if orderedIndexes[i] != participant]
    Xtest, Ytest = [x for i, x in enumerate(X) if orderedIndexes[i] == participant], [x for i, x in enumerate(Y) if orderedIndexes[i] == participant]

    criterion = nn.BCELoss() 
    optimizer = optim.Adam(model2.parameters(), lr=0.01)
    
    K = 5
    def ordinal_target(y, K):

        if not isinstance(y, torch.Tensor):
            y = torch.tensor(y, dtype=torch.long)

        y = y.view(-1) 
        cum_labels = torch.zeros(y.size(0), K - 1)
        for k in range(K - 1):
            cum_labels[:, k] = (y > k).float().squeeze()
        return cum_labels
    
    y_train_cum = ordinal_target(Ytrain, K)
    
    model2.train()
    for epoch in range(10):
        optimizer.zero_grad()
        Xtrain = torch.tensor(Xtrain, dtype=torch.float)
        outputs = model2(Xtrain)
        loss = criterion(outputs, y_train_cum)
        loss.backward()
        optimizer.step()
    
    model2.eval()
    with torch.no_grad():
        cum_probs = model2(torch.tensor(Xtest, dtype=torch.float))
        Ypred = cum_probs_to_labels(cum_probs)
    
    y_true_all.extend(torch.tensor(Ytest).numpy())
    y_pred_all.extend(Ypred.numpy())

    results.append((pearsonr(y_true_all, y_pred_all)[0], CCcoefficient(y_true_all, y_pred_all)))

print(results)

    

In [ ]:
indexes = data.index

participants = list(set(indexes))

X = [element[0] for element in valencePercentileClasses]
Y = [element[1] for element in valencePercentileClasses]
orderedIndexes = [element[2] for element in valencePercentileClasses]

y_true_all = []
y_pred_all = []

results = []

for participant in participants:
    
    trainIdx = [p for p in participants if p != participant]
    testIdx = participant

    Xtrain, Ytrain = [x for i, x in enumerate(X) if orderedIndexes[i] != participant], [x for i, x in enumerate(Y) if orderedIndexes[i] != participant]
    Xtest, Ytest = [x for i, x in enumerate(X) if orderedIndexes[i] == participant], [x for i, x in enumerate(Y) if orderedIndexes[i] == participant]

    y_train_cum = ordinal_target(Ytrain, 5)

    optimizer = optim.Adam(model3.parameters(), lr=0.01)
    criterion = nn.BCELoss()

    def ordinal_target(y, K):
        if not isinstance(y, torch.Tensor):
            y = torch.tensor(y, dtype=torch.long)
        y = y.view(-1)
        cum_labels = torch.zeros(y.size(0), K - 1)
        for k in range(K - 1):
            cum_labels[:, k] = (y > k).float()
        return cum_labels

    for epoch in range(10):
        model3.train()
        optimizer.zero_grad()
        Xtrain = torch.tensor(Xtrain, dtype=torch.float)
        outputs = model3(Xtrain.unsqueeze(1))
        loss = criterion(outputs, y_train_cum)
        loss.backward()
        optimizer.step()

    model3.eval()
    with torch.no_grad():
        cum_probs = model3(torch.tensor(Xtest, dtype=torch.float).unsqueeze(1))
        y_pred = cum_probs_to_labels(cum_probs)

    y_true_all.extend(torch.tensor(Ytest).numpy())
    y_pred_all.extend(y_pred.numpy())

    results.append((pearsonr(y_true_all, y_pred_all)[0], CCcoefficient(y_true_all, y_pred_all)))

print(results)